In [1]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json

from textwrap import dedent

from pathlib import Path


from time import perf_counter

# Preliminari

In [2]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_2
TEMPERATURE = 0.0
MODEL = constants.OPENAI_GPT_4_1_NANO

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, constants.AnnotationsExtended)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è estrarre le caratteristiche strutturate dal referto RM fornito e mappare i dati nello schema JSON sottostante.

Regole di output rigorose:

1. Restituisci esclusivamente un oggetto JSON valido. Nessun testo introduttivo, spiegazione, codice o commento fuori dal JSON.
2. Rispetta esattamente i tipi e i valori consentiti per ciascun campo.
4. Se un campo richiede valori enumerati, usa esclusivamente uno dei valori ammessi, senza mai usare sinonimi o varianti.
5. Se un campo è numerico ma nel testo non compare un valore, scrivi null, come indicato nello schema.
6. Non aggiungere o inventare informazioni non presenti nel referto.

Ecco dei criteri e consigli di estrazione per alcuni dei campi che dovrai estrarre:

Morfologia:
  - Se il referto menziona componenti aggettanti, vegetanti o endoluminali, classifica come solido_polipoide.
  - Se si parla di is

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

train_data, validation_data = data['train'], data['validation']

################################
# Convert float columns to Int64
################################
float_cols = train_data.select_dtypes("float").columns
for col in float_cols:
    train_data[col] = train_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(train_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")

len(train_data) = 185
len(validation_data) = 63


# Upload data to Openai dashboard

## Create local fine-tuning datasets